## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of LangGraph's core components, including StateGraph, MessageGraph, nodes, and edges.
* Design and implement complex, stateful agentic workflows using conditional routing and ReAct patterns.
* Integrate external tools and LLMs effectively within a LangGraph agent to solve multi-step problems.
* Apply best practices for structuring, debugging, and optimizing LangGraph agents for robustness and efficiency.
* Solve a capstone problem by building a sophisticated AI agent that leverages multiple LangGraph features to achieve a specific goal.


## Final Assessment: Building Robust AI Agents with LangGraph

Welcome to the final assessment for GS-01: Building Your First AI Agent with LangGraph! This assessment is designed to evaluate your comprehensive understanding and practical application of the concepts covered throughout this course. By 2026, the ability to orchestrate complex, stateful AI workflows is paramount for developing truly intelligent and autonomous agents. LangGraph stands at the forefront of this evolution, providing the necessary primitives for robust agentic design.

This assessment will challenge you to:

1.  **Recall and explain core LangGraph concepts**: From the fundamental `StateGraph` and `MessageGraph` to the nuances of conditional edges and state management, you should be able to articulate *why* and *how* these components are used.
2.  **Design and implement sophisticated agentic loops**: You will demonstrate your ability to create multi-turn, decision-making agents that can adapt their behavior based on intermediate results, mimicking advanced ReAct (Reasoning and Acting) patterns.
3.  **Integrate and manage external tools**: A key aspect of modern AI agents is their ability to interact with the outside world. You will show proficiency in incorporating various tools (e.g., search, data analysis, code execution) into your LangGraph workflows.
4.  **Problem-solve with LangGraph**: The capstone project will require you to apply all learned concepts to build a functional, multi-step AI agent that addresses a specific, practical problem. This includes defining appropriate state, designing efficient nodes, and implementing intelligent routing logic.

Success in this assessment signifies your readiness to tackle real-world agent development challenges, building scalable and reliable AI systems that can perform complex tasks autonomously. Good luck!


### Review Questions

Answer the following questions to test your theoretical understanding of LangGraph and agentic design principles.

1.  **StateGraph vs. MessageGraph**: Explain the primary difference between `StateGraph` and `MessageGraph` in LangGraph. When would you choose one over the other, and what are the implications for state management?
2.  **Conditional Edges**: Describe the mechanism of `add_conditional_edges`. Provide a conceptual example of a scenario where a conditional edge is essential for an agent's decision-making process.
3.  **ReAct Pattern in LangGraph**: How can the ReAct (Reasoning and Acting) pattern be implemented effectively using LangGraph? Outline the typical nodes and edges involved in a ReAct loop.
4.  **Tool Integration**: You're building an agent that needs to perform web searches and then summarize the results. Describe the steps to integrate a `search_tool` into your LangGraph agent, including how the agent's state would typically manage the tool's output.
5.  **Handling Errors and Retries**: In a production-grade LangGraph agent, how would you approach error handling (e.g., a tool call failing, an LLM call timing out)? Discuss potential strategies for retries or fallback mechanisms within the graph structure.
6.  **Graph Compilation**: Explain the significance of `graph.compile()` in LangGraph. What benefits does it provide, especially in terms of performance and deployment?
7.  **Agent State Management**: Consider an agent that needs to maintain a conversation history, a list of facts gathered, and a current goal. How would you define the `State` object for such an agent, and how would different nodes interact with and update this state?


### Capstone Project: The 'Market Research Analyst' Agent

**Problem Description:**

Your task is to build a sophisticated AI agent using LangGraph that acts as a 'Market Research Analyst'. This agent should be able to take a user query about a specific market trend or company, perform research, analyze the findings, and generate a concise report. The agent must demonstrate intelligent decision-making, tool usage, and state management.

**Agent Requirements:**

1.  **Initial Query**: The agent receives an initial user query (e.g., "Analyze the market trends for AI-powered healthcare diagnostics in Q1 2026.").
2.  **Research Phase**: The agent must use a `search_tool` to gather relevant information. It should be able to formulate search queries based on the user's request.
3.  **Analysis Phase**: After gathering information, the agent needs to analyze the search results. This involves using an LLM to extract key insights, identify trends, and synthesize information. It should also be able to identify if more research is needed.
4.  **Report Generation**: Once sufficient information is gathered and analyzed, the agent should compile a concise, well-structured report summarizing the findings.
5.  **Conditional Logic**: The agent must intelligently decide whether to:
    *   Perform more searches if initial results are insufficient or unclear.
    *   Proceed to analysis if enough data is available.
    *   Generate the final report.
6.  **State Management**: The agent's state should effectively track:
    *   The original user query.
    *   Search queries made and their results.
    *   Analyzed insights.
    *   The final report content.
    *   A flag indicating if the task is complete.
7.  **Tool Usage**: You will need to define at least one custom tool (e.g., a simulated `search_tool`).

**Deliverables:**

*   A complete LangGraph implementation of the 'Market Research Analyst' agent.
*   Clear comments explaining each part of your code.
*   Demonstrate the agent's functionality with at least two distinct user queries.

**Evaluation Criteria:**

*   Correctness and completeness of the LangGraph implementation.
*   Effective use of `StateGraph`, nodes, and conditional edges.
*   Intelligent decision-making and routing logic.
*   Proper state management.
*   Clarity and readability of the code and comments.
*   Ability to successfully generate a relevant report for given queries.


In [ ]:
import operator
from typing import Annotated, List, Tuple, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# --- Define Agent State ---
# This defines the state of our agent. It's a TypedDict that will be passed between nodes.
class AgentState(TypedDict):
    query: str
    chat_history: Annotated[List[BaseMessage], operator.add]
    search_results: List[str]
    analysis_results: str
    report: str
    needs_more_research: bool

# --- Define Tools (Simulated for Assessment) ---
@tool
def search_tool(query: str) -> str:
    """Simulates a web search for market research. Returns relevant information."""
    print(f"\n--- Performing simulated search for: {query} ---")
    # In a real scenario, this would call a search API (e.g., Tavily, Google Search)
    if "AI-powered healthcare diagnostics" in query:
        return "Key trends: increased investment in predictive analytics, regulatory challenges, demand for personalized medicine. Companies: Siemens Healthineers, GE Healthcare, Philips. Q1 2026 saw 15% growth in venture capital for this sector."
    elif "quantum computing market" in query:
        return "Emerging trends: focus on error correction, hybrid quantum-classical algorithms. Major players: IBM, Google, Microsoft. Market size projected to reach $1.5B by 2030. Q1 2026 saw significant breakthroughs in qubit stability."
    elif "electric vehicle battery technology" in query:
        return "Innovations: solid-state batteries, silicon anodes, faster charging. Challenges: range anxiety, charging infrastructure. Companies: CATL, Panasonic, LG Energy Solution. Q1 2026 saw new partnerships for gigafactory development."
    else:
        return f"No specific results found for '{query}'. General market overview: innovation continues across tech sectors."

# --- Initialize LLM ---
# Ensure you have your OPENAI_API_KEY set in your environment variables.
# For 2026, we might use more advanced models or local LLMs via Ollama/LiteLLM.
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

# --- Define Nodes (Functions that operate on the state) ---
def research_node(state: AgentState) -> AgentState:
    """Performs research based on the query and updates search_results."""
    print("\n--- Entering Research Node ---")
    current_query = state["query"]
    # Use LLM to refine search query if needed, or directly use the main query
    refined_search_query = llm.invoke(
        f"Given the user's request: '{current_query}', what would be the most effective search query to find relevant market research information? Respond with only the search query." 
    ).content
    
    results = search_tool.invoke({"query": refined_search_query})
    
    # Update chat history for context in future LLM calls
    state["chat_history"].append(HumanMessage(content=f"Performed search for '{refined_search_query}'. Results: {results}"))
    state["search_results"].append(results)
    
    # Decide if more research is needed based on initial results (simplified for assessment)
    if "No specific results found" in results and len(state["search_results"]) < 2: # Allow one retry
        state["needs_more_research"] = True
        state["query"] = "Refine search for: " + current_query # Modify query for next research attempt
    else:
        state["needs_more_research"] = False
        
    return state

def analysis_node(state: AgentState) -> AgentState:
    """Analyzes search results and updates analysis_results and needs_more_research flag."""
    print("\n--- Entering Analysis Node ---")
    combined_results = "\n".join(state["search_results"])
    
    analysis_prompt = f"""You are a market research analyst. Analyze the following search results related to '{state["query"]}'.
    
    Search Results:
    {combined_results}
    
    Identify key trends, major players, challenges, and any significant statistics or projections. Also, determine if the information is sufficient to generate a comprehensive report or if more research is needed. If more research is needed, state 'MORE_RESEARCH_NEEDED' at the beginning of your response, followed by a brief explanation. Otherwise, provide a detailed analysis.
    """
    
    analysis_output = llm.invoke(analysis_prompt).content
    
    if analysis_output.startswith("MORE_RESEARCH_NEEDED"):
        state["needs_more_research"] = True
        state["analysis_results"] = analysis_output.replace("MORE_RESEARCH_NEEDED", "").strip()
        state["query"] = f"Further research needed on: {state["query"]}. Reason: {state["analysis_results"]}"
    else:
        state["needs_more_research"] = False
        state["analysis_results"] = analysis_output
        
    state["chat_history"].append(HumanMessage(content=f"Analysis performed. More research needed: {state['needs_more_research']}"))
    return state

def report_generation_node(state: AgentState) -> AgentState:
    """Generates the final report based on analysis results."""
    print("\n--- Entering Report Generation Node ---")
    report_prompt = f"""Based on the following analysis for '{state["query"]}', generate a concise, professional market research report.
    
    Analysis:
    {state["analysis_results"]}
    
    The report should include an executive summary, key findings, market trends, major players, and a conclusion. Format it clearly with headings.
    """
    
    final_report = llm.invoke(report_prompt).content
    state["report"] = final_report
    state["chat_history"].append(HumanMessage(content="Final report generated."))
    return state

# --- Define Conditional Edges (Router Functions) ---
def decide_next_step(state: AgentState) -> str:
    """Decides whether to do more research, analyze, or generate report."""
    print("\n--- Deciding Next Step ---")
    if state["needs_more_research"] and len(state["search_results"]) < 3: # Limit research iterations
        print("Decision: More Research Needed.")
        return "research"
    elif state["analysis_results"] and not state["needs_more_research"]:
        print("Decision: Generate Report.")
        return "report_generation"
    else:
        print("Decision: Proceed to Analysis.")
        return "analysis"

# --- Build the LangGraph Workflow ---
# Initialize the StateGraph with our defined state
workflow = StateGraph(AgentState)

# Add nodes to the workflow
workflow.add_node("research", research_node)
workflow.add_node("analysis", analysis_node)
workflow.add_node("report_generation", report_generation_node)

# Set the entry point
workflow.set_entry_point("research")

# Add edges
# After research, decide if more research is needed or move to analysis
workflow.add_conditional_edges(
    "research",
    decide_next_step, # This function determines the next node
    {
        "research": "research", # Loop back to research if more is needed
        "analysis": "analysis"   # Move to analysis if research is sufficient
    }
)

# After analysis, decide if more research is needed or move to report generation
workflow.add_conditional_edges(
    "analysis",
    decide_next_step, # This function determines the next node
    {
        "research": "research",         # Loop back to research if analysis indicates more data is needed
        "report_generation": "report_generation" # Move to report generation if analysis is complete
    }
)

# After report generation, the task is complete
workflow.add_edge("report_generation", END)

# Compile the graph
app = workflow.compile()

# --- Test the Agent ---
print("\n--- Testing Agent with Query 1 ---")
initial_state_1 = {
    "query": "Analyze the market trends for AI-powered healthcare diagnostics in Q1 2026.",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

final_state_1 = app.invoke(initial_state_1)
print("\n--- Final Report 1 ---")
print(final_state_1["report"])

print("\n\n--- Testing Agent with Query 2 ---")
initial_state_2 = {
    "query": "Provide a market overview for quantum computing in 2026, focusing on key players and emerging technologies.",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

final_state_2 = app.invoke(initial_state_2)
print("\n--- Final Report 2 ---")
print(final_state_2["report"])

print("\n\n--- Testing Agent with Query 3 (Simulated Insufficient Data) ---")
initial_state_3 = {
    "query": "What are the latest advancements in sustainable urban planning technologies in Q1 2026?",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

final_state_3 = app.invoke(initial_state_3)
print("\n--- Final Report 3 ---")
print(final_state_3["report"])


In [ ]:
import operator
from typing import Annotated, List, Tuple, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# --- Define Agent State ---
# The state defines the information that flows between different nodes in our graph.
# Annotated[List[BaseMessage], operator.add] is used for chat_history to automatically append new messages.
class AgentState(TypedDict):
    query: str  # The original user query
    chat_history: Annotated[List[BaseMessage], operator.add] # Conversation history for context
    search_results: List[str] # Accumulated search results
    analysis_results: str # The output of the analysis phase
    report: str # The final generated report
    needs_more_research: bool # Flag to control conditional routing

# --- Define Tools (Simulated for Assessment) ---
# In a real-world scenario, these would be actual API calls or complex functions.
@tool
def search_tool(query: str) -> str:
    """Simulates a web search for market research. Returns relevant information."""
    print(f"\n--- Performing simulated search for: {query} ---")
    # This is a simplified mock. A real tool would use libraries like TavilySearch, GoogleSearchAPIWrapper, etc.
    if "AI-powered healthcare diagnostics" in query.lower():
        return "Key trends: increased investment in predictive analytics, regulatory challenges, demand for personalized medicine. Companies: Siemens Healthineers, GE Healthcare, Philips. Q1 2026 saw 15% growth in venture capital for this sector. Source: Industry Report 2026."
    elif "quantum computing market" in query.lower():
        return "Emerging trends: focus on error correction, hybrid quantum-classical algorithms. Major players: IBM, Google, Microsoft. Market size projected to reach $1.5B by 2030. Q1 2026 saw significant breakthroughs in qubit stability. Source: TechCrunch 2026."
    elif "electric vehicle battery technology" in query.lower():
        return "Innovations: solid-state batteries, silicon anodes, faster charging. Challenges: range anxiety, charging infrastructure. Companies: CATL, Panasonic, LG Energy Solution. Q1 2026 saw new partnerships for gigafactory development. Source: EV World News."
    elif "sustainable urban planning technologies" in query.lower():
        # Simulate a case where initial search might be less specific
        return "General information on urban planning: smart city initiatives, green infrastructure, public transport optimization. No specific Q1 2026 tech advancements found in initial search."
    else:
        return f"No specific results found for '{query}'. General market overview: innovation continues across tech sectors. Try refining your query."

# --- Initialize LLM ---
# We use ChatOpenAI as a powerful, general-purpose LLM. 
# Ensure OPENAI_API_KEY is set in your environment variables.
# For 2026, consider using models like GPT-4o, Claude 3 Opus, or specialized fine-tuned models.
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

# --- Define Nodes (Functions that operate on the state) ---
# Each node takes the current state, performs an action, and returns an updated state.

def research_node(state: AgentState) -> AgentState:
    """Performs research based on the query and updates search_results."""
    print("\n--- Entering Research Node ---")
    current_query = state["query"]
    
    # Use LLM to refine the search query for better results. This is a common agentic pattern.
    refined_search_query_response = llm.invoke(
        f"Given the user's request: '{current_query}', what would be the most effective and concise search query to find relevant market research information? Respond with only the search query, no extra text." 
    ).content
    
    # Invoke the simulated search tool
    results = search_tool.invoke({"query": refined_search_query_response})
    
    # Update the state with new information
    state["chat_history"].append(HumanMessage(content=f"Performed search for '{refined_search_query_response}'. Results: {results}"))
    state["search_results"].append(results)
    
    # Simple logic to decide if more research might be needed based on the search output
    # In a real agent, this might involve more sophisticated parsing or LLM-based evaluation.
    if "No specific results found" in results or "General information" in results:
        state["needs_more_research"] = True
        # Modify the query for the next research attempt to encourage refinement
        state["query"] = f"Refine search for: {current_query}. Initial results were too general or empty."
    else:
        state["needs_more_research"] = False
        
    return state

def analysis_node(state: AgentState) -> AgentState:
    """Analyzes search results and updates analysis_results and needs_more_research flag."""
    print("\n--- Entering Analysis Node ---")
    combined_results = "\n---\n".join(state["search_results"])
    
    analysis_prompt = f"""You are a market research analyst. Analyze the following search results related to the original query: '{state["query"]}'.
    
    Search Results:
    {combined_results}
    
    Identify key trends, major players, challenges, and any significant statistics or projections. 
    
    Crucially, determine if the information is sufficient to generate a comprehensive report. 
    If you believe more research is absolutely needed to provide a good report, start your response with 'MORE_RESEARCH_NEEDED:' followed by a brief explanation of what specific information is missing or unclear.
    Otherwise, provide a detailed analysis ready for report generation.
    """
    
    analysis_output = llm.invoke(analysis_prompt).content
    
    if analysis_output.startswith("MORE_RESEARCH_NEEDED:"):
        state["needs_more_research"] = True
        state["analysis_results"] = analysis_output.replace("MORE_RESEARCH_NEEDED:", "").strip()
        # Update the query to guide the next research iteration based on analysis feedback
        state["query"] = f"Further research needed on: {state["query"]}. Reason: {state["analysis_results"]}"
    else:
        state["needs_more_research"] = False
        state["analysis_results"] = analysis_output
        
    state["chat_history"].append(HumanMessage(content=f"Analysis performed. More research needed: {state['needs_more_research']}"))
    return state

def report_generation_node(state: AgentState) -> AgentState:
    """Generates the final report based on analysis results."""
    print("\n--- Entering Report Generation Node ---")
    report_prompt = f"""Based on the following analysis for the market research query: '{state["query"]}', generate a concise, professional market research report.
    
    Analysis:
    {state["analysis_results"]}
    
    The report should include an Executive Summary, Key Findings, Market Trends, Major Players, and a Conclusion. Format it clearly with headings and bullet points where appropriate. Ensure it's professional and easy to read.
    """
    
    final_report = llm.invoke(report_prompt).content
    state["report"] = final_report
    state["chat_history"].append(HumanMessage(content="Final report generated."))
    return state

# --- Define Conditional Edges (Router Functions) ---
# These functions determine the next node to execute based on the current state.

def decide_next_step(state: AgentState) -> str:
    """Decides whether to do more research, analyze, or generate report based on state flags."""
    print("\n--- Deciding Next Step ---")
    # If more research is needed and we haven't exceeded a retry limit (e.g., 3 searches)
    if state["needs_more_research"] and len(state["search_results"]) < 3: 
        print("Decision: More Research Needed. Looping back to research.")
        return "research"
    # If analysis is complete and no more research is needed, generate the report
    elif state["analysis_results"] and not state["needs_more_research"]:
        print("Decision: Analysis complete. Proceeding to Report Generation.")
        return "report_generation"
    # Otherwise, if we have search results but haven't analyzed them sufficiently, proceed to analysis
    else:
        print("Decision: Proceeding to Analysis.")
        return "analysis"

# --- Build the LangGraph Workflow ---
# Initialize the StateGraph with our defined AgentState
workflow = StateGraph(AgentState)

# Add nodes to the workflow, mapping a name to each function
workflow.add_node("research", research_node)
workflow.add_node("analysis", analysis_node)
workflow.add_node("report_generation", report_generation_node)

# Set the entry point for the graph execution
workflow.set_entry_point("research")

# Add conditional edges to create dynamic routing
# From 'research' node, decide whether to loop back for more research or move to 'analysis'
workflow.add_conditional_edges(
    "research",
    decide_next_step, # The router function
    {
        "research": "research", # If decide_next_step returns "research", go back to "research" node
        "analysis": "analysis"   # If decide_next_step returns "analysis", go to "analysis" node
    }
)

# From 'analysis' node, decide whether to go back to 'research' (if more data is needed) 
# or move to 'report_generation' (if analysis is complete)
workflow.add_conditional_edges(
    "analysis",
    decide_next_step, # The router function
    {
        "research": "research",         # If decide_next_step returns "research", go back to "research" node
        "report_generation": "report_generation" # If decide_next_step returns "report_generation", go to "report_generation" node
    }
)

# After 'report_generation', the task is complete, so we end the graph execution
workflow.add_edge("report_generation", END)

# Compile the graph for execution. This optimizes the graph for performance.
app = workflow.compile()

# --- Test the Agent ---
print("\n--- Testing Agent with Query 1: AI-powered Healthcare Diagnostics ---")
initial_state_1 = {
    "query": "Analyze the market trends for AI-powered healthcare diagnostics in Q1 2026.",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

# Invoke the agent with the initial state
final_state_1 = app.invoke(initial_state_1)
print("\n--- Final Report 1 ---")
print(final_state_1["report"])

print("\n\n--- Testing Agent with Query 2: Quantum Computing Market ---")
initial_state_2 = {
    "query": "Provide a market overview for quantum computing in 2026, focusing on key players and emerging technologies.",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

final_state_2 = app.invoke(initial_state_2)
print("\n--- Final Report 2 ---")
print(final_state_2["report"])

print("\n\n--- Testing Agent with Query 3: Sustainable Urban Planning (Simulated Insufficient Data initially) ---")
initial_state_3 = {
    "query": "What are the latest advancements in sustainable urban planning technologies in Q1 2026?",
    "chat_history": [],
    "search_results": [],
    "analysis_results": "",
    "report": "",
    "needs_more_research": False
}

final_state_3 = app.invoke(initial_state_3)
print("\n--- Final Report 3 ---")
print(final_state_3["report"])
